In [0]:
import os, sys
import pandas as pd

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, repo_root)

from src import config_loader as cl
from src import silver as sv
from src import scd2
from src import meta

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("resource", "ALL", "Resource (ALL or e.g. Patient)")
dbutils.widgets.text("load_dates", "", "Bronze load dates, comma separated (blank = all)")
dbutils.widgets.text("catalog", "", "Catalog override (optional)")
dbutils.widgets.text("run_id", "", "Run id (blank = generate)")
dbutils.widgets.text("pipeline_name", "manual_silver", "Pipeline name")

g = lambda k: dbutils.widgets.get(k).strip()
load_dates = [d.strip() for d in g("load_dates").split(",") if d.strip()] or None
run_id = g("run_id") or cl.new_batch_id()
pipeline_name = g("pipeline_name")
overrides = {"project": {"catalog": g("catalog")}} if g("catalog") else None
cfg = cl.load_config(os.path.join(repo_root, "config", "config.yaml"), overrides)

only = None if g("resource").upper() == "ALL" else [g("resource")]
resources = cl.get_resources(cfg, only)
print(f"run_id={run_id} | resources={[r['name'] for r in resources]} | load_dates={load_dates or 'ALL'}")

In [0]:
meta.create_meta_tables(spark, cfg)

results = []
for r in resources:                      # Patient -> Encounter -> Observation -> Condition
    n = r["name"]
    t0 = cl.utc_now()
    status, error, res = "SUCCESS", None, {}
    try:
        df = sv.build_silver_df(spark, cfg, n, load_dates)
        res = scd2.scd2_merge(spark, cfg, n, df)
    except Exception as e:
        status, error = "FAILED", f"{type(e).__name__}: {e}"
    t1 = cl.utc_now()

    meta.log_run(spark, cfg, run_id=run_id, pipeline_name=pipeline_name, stage="SILVER",
                 resource_type=n, load_date=None, status=status, source_format="json",
                 started_at=t0, ended_at=t1,
                 data_saved_at=t1 if status == "SUCCESS" else None,
                 records_read=res.get("incoming_versions"),
                 records_written=res.get("new_versions_inserted"), error_message=error)
    results.append({"resource": n, "status": status,
                    **{k: v for k, v in res.items() if k not in ("resource", "table")},
                    "error": error})

res_df = pd.DataFrame(results)
display(res_df)

bad = res_df[res_df["status"] != "SUCCESS"]
if len(bad):
    raise Exception(f"Silver SCD2 problems:\n{bad.to_string()}")

In [0]:
def q(sql):
    return spark.sql(sql).collect()[0][0]

checks = []
for r in resources:
    n, key = r["name"], r["silver_key"]
    t = cl.resource_table(cfg, "silver", n)
    checks += [
        {"check": f"{n}: ids without exactly one current row", "expected": 0,
         "result": q(f"SELECT count(*) FROM (SELECT {key} FROM {t} GROUP BY {key} "
                     f"HAVING sum(CAST(is_current AS INT)) <> 1)")},
        {"check": f"{n}: duplicate (id, version_number)", "expected": 0,
         "result": q(f"SELECT count(*) FROM (SELECT {key}, version_number FROM {t} "
                     f"GROUP BY 1, 2 HAVING count(*) > 1)")},
        {"check": f"{n}: gaps or overlaps between versions", "expected": 0,
         "result": q(f"""SELECT count(*) FROM (
                SELECT effective_to,
                       lead(effective_from) OVER (PARTITION BY {key} ORDER BY version_number) AS nxt
                FROM {t})
              WHERE (nxt IS NOT NULL AND (effective_to IS NULL OR effective_to <> nxt))
                 OR (nxt IS NULL AND effective_to IS NOT NULL)""")},
        {"check": f"{n}: current rows that have an effective_to", "expected": 0,
         "result": q(f"SELECT count(*) FROM {t} WHERE is_current AND effective_to IS NOT NULL")},
    ]
chk = pd.DataFrame(checks)
chk["ok"] = chk["result"] == chk["expected"]
display(chk)

In [0]:
summary = []
for r in resources:
    n, key = r["name"], r["silver_key"]
    t = cl.resource_table(cfg, "silver", n)
    summary.append({"table": t,
        "total_rows": q(f"SELECT count(*) FROM {t}"),
        "distinct_ids": q(f"SELECT count(DISTINCT {key}) FROM {t}"),
        "current_rows": q(f"SELECT count(*) FROM {t} WHERE is_current"),
        "ids_with_history": q(f"SELECT count(*) FROM (SELECT {key} FROM {t} GROUP BY {key} HAVING count(*) > 1)")})
display(pd.DataFrame(summary))

t = cl.resource_table(cfg, "silver", "Patient")
display(spark.sql(f"""SELECT patient_id, version_number, is_current, effective_from, effective_to,
                             gender, birth_date, city, meta_version_id, extraction_timestamp, load_date
                      FROM {t} ORDER BY effective_from DESC LIMIT 10"""))